In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Within-Region (Within-Electrode) PAC Analysis

Hit/miss-aware trial exports and high-PAC LF/HF phase-locking plots.

In [ ]:
"""
Import required libraries for PAC analysis
"""
import numpy as np
import mne
from tensorpac import Pac
from mne_connectivity import seed_target_indices
import matplotlib.pyplot as plt
import pandas as pd
import warnings
import os
from scipy.signal import hilbert

from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df
from membandit.granger import build_hit_miss_trial_map

# Statistical functions for optional cluster tests
from mne.stats import (
    permutation_cluster_test,
    permutation_cluster_1samp_test,
    ttest_ind_no_p,
    ttest_1samp_no_p,
)

warnings.filterwarnings('ignore')


In [ ]:
"""
Add custom analysis utilities to path and import
"""
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')
from LFPAnalysis import oscillation_utils


In [ ]:
"""
Data loading and subject selection
"""
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

subj_ids = subj_df.MSSMCode.astype(str).values
print(f'Loaded {len(subj_ids)} subjects for analysis')


In [ ]:
"""
Load and preprocess behavior tables via mv_granger_utils
"""
learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print(f'learn_df shape: {learn_df.shape}')
print(f'combined_df shape: {combined_df.shape}')
print('combined required columns present:', {'participant', 'condition', 'trials_gamble', 'hits'}.issubset(combined_df.columns))


In [ ]:
"""
Analysis configuration
"""
evs = {
    'gamble_start': [-1.0, 0.0],
    'feedback_start': [0.0, 1.5],
    'baseline_start': [0.0, 0.75],
    'cue_on': [0.0, 1.0],
}
analysis_evs = ['cue_on']
baseline_ev = 'baseline_start'
day = 'Day1'

for ev in analysis_evs:
    if ev not in evs:
        raise ValueError(f'Analysis event "{ev}" is not defined in evs')

print('Analysis events:', analysis_evs)
print('Day:', day)


In [ ]:
"""
Frequency band configuration
"""
phase_frequency = 'theta'
f_pha_start, f_pha_end = 4, 8
f_pha_nb = 20
f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
f_pha = np.c_[f_pha_mid - f_pha_mid / 4.0, f_pha_mid + f_pha_mid / 4.0]

amp_frequency = 'hfa'
f_amp_start, f_amp_end = 70, 200
f_amp_nb = 10
f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
f_amp = np.c_[f_amp_mid - f_amp_mid / 8.0, f_amp_mid + f_amp_mid / 8.0]

print(f'Phase band: {phase_frequency} ({f_pha_start}-{f_pha_end} Hz, {f_pha_nb} bins)')
print(f'Amplitude band: {amp_frequency} ({f_amp_start}-{f_amp_end} Hz, {f_amp_nb} bins)')


In [ ]:
"""
Output directory configuration
"""
save_dir_data = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac'
save_dir_plots = '/Users/salmanqasim/Documents/Projects/MemBanditsEMU/Plots/MemoryBandit-WithinRegionPAC'
os.makedirs(save_dir_data, exist_ok=True)
os.makedirs(save_dir_plots, exist_ok=True)

print('Data save dir:', save_dir_data)
print('Plot save dir:', save_dir_plots)


In [ ]:
"""
Helper functions for PAC computation, hit/miss mapping, and high-PAC visualization
"""

def compute_pac_for_pair(pac_obj, ph_sig, amp_sig, sfreq, edges, n_perm=250, random_state=0):
    epoch_pac = pac_obj.filterfit(
        sfreq,
        ph_sig,
        amp_sig,
        n_perm=n_perm,
        n_jobs=-1,
        edges=edges,
        random_state=random_state,
    )

    surr_mean = pac_obj.surrogates.mean(axis=0)
    surr_std = pac_obj.surrogates.std(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        z_scored_pac = (epoch_pac - surr_mean) / surr_std

    return z_scored_pac, epoch_pac


def create_pac_dataframe(
    z_scored_pac,
    subj_id,
    subj_info,
    phase_roi,
    amp_roi,
    event,
    phase_freq,
    amp_freq,
    ph_ch_name,
    amp_ch_name,
    hemi,
    learn_df,
):
    avg_conn = np.nanmean(z_scored_pac, axis=(0, 1))

    z_scored_pac_sig = z_scored_pac.copy()
    z_scored_pac_sig[np.abs(z_scored_pac_sig) <= 1.96] = np.nan
    sum_conn = np.nansum(z_scored_pac_sig, axis=(0, 1))

    pwise_df = pd.DataFrame({
        'participant': subj_id,
        'age': subj_info['Age'],
        'sex': subj_info['Sex'].strip() if isinstance(subj_info['Sex'], str) else subj_info['Sex'],
        'amp_roi': amp_roi,
        'phase_roi': phase_roi,
        'event': event,
        'band0': phase_freq,
        'band1': amp_freq,
        'trials': np.arange(1, len(sum_conn) + 1),
        'label': f'{ph_ch_name}:{amp_ch_name}',
        'phase_channel': ph_ch_name,
        'amp_channel': amp_ch_name,
        'hemi': hemi,
        'avg_conn': avg_conn,
        'sum_conn': sum_conn,
    })

    subj_learn_df = learn_df[learn_df.participant == subj_id].copy()
    pwise_df = pwise_df.merge(subj_learn_df, on=['participant', 'trials'], how='left')

    if 'rpe' in pwise_df.columns:
        denom = 2.0 * np.nanstd(pwise_df['rpe'])
        pwise_df['zrpe'] = (pwise_df['rpe'] - np.nanmean(pwise_df['rpe'])) / denom if np.isfinite(denom) and denom > 0 else np.nan
    if 'DPRIME' in pwise_df.columns:
        denom = 2.0 * np.nanstd(pwise_df['DPRIME'])
        pwise_df['zpm'] = (pwise_df['DPRIME'] - np.nanmean(pwise_df['DPRIME'])) / denom if np.isfinite(denom) and denom > 0 else np.nan

    return pwise_df


def compute_high_pac_mask(trial_pac_z, hit_miss, high_pac_quantile=0.9, min_high_trials=5):
    trial_pac_z = np.asarray(trial_pac_z, dtype=float)
    hit_miss = np.asarray(hit_miss, dtype=object)
    mask = np.zeros(trial_pac_z.shape[0], dtype=bool)

    for cond in ('hit', 'miss'):
        cond_mask = (hit_miss == cond) & np.isfinite(trial_pac_z)
        n_cond = int(cond_mask.sum())
        if n_cond == 0:
            continue

        cond_scores = trial_pac_z[cond_mask]
        threshold = np.nanquantile(cond_scores, high_pac_quantile)
        cond_selected = np.where(cond_mask & (trial_pac_z >= threshold))[0]

        min_take = min(min_high_trials, n_cond)
        if len(cond_selected) < min_take:
            cond_indices = np.where(cond_mask)[0]
            rank_order = np.argsort(cond_scores)[::-1]
            cond_selected = cond_indices[rank_order[:min_take]]

        mask[cond_selected] = True

    return mask


def summarize_hit_miss_pair(pwise_df):
    valid_df = pwise_df[pwise_df['alignment_valid'] == True].copy()

    hit_df = valid_df[valid_df['hit_miss'] == 'hit']
    miss_df = valid_df[valid_df['hit_miss'] == 'miss']

    mean_avg_hit = float(np.nanmean(hit_df['avg_conn'])) if len(hit_df) > 0 else np.nan
    mean_avg_miss = float(np.nanmean(miss_df['avg_conn'])) if len(miss_df) > 0 else np.nan
    mean_sum_hit = float(np.nanmean(hit_df['sum_conn'])) if len(hit_df) > 0 else np.nan
    mean_sum_miss = float(np.nanmean(miss_df['sum_conn'])) if len(miss_df) > 0 else np.nan

    return {
        'mean_avg_conn_hit': mean_avg_hit,
        'mean_avg_conn_miss': mean_avg_miss,
        'mean_sum_conn_hit': mean_sum_hit,
        'mean_sum_conn_miss': mean_sum_miss,
        'delta_hit_minus_miss_avg_conn': mean_avg_hit - mean_avg_miss if np.isfinite(mean_avg_hit) and np.isfinite(mean_avg_miss) else np.nan,
        'delta_hit_minus_miss_sum_conn': mean_sum_hit - mean_sum_miss if np.isfinite(mean_sum_hit) and np.isfinite(mean_sum_miss) else np.nan,
    }


def plot_high_pac_lfp_phase_locking(
    phase_sig,
    amp_sig,
    trial_pac_z,
    hit_miss_labels,
    sfreq,
    times,
    f_pha_band,
    f_amp_band,
    out_file,
    title_prefix,
    high_pac_quantile=0.9,
    min_high_trials=5,
):
    phase_sig = np.asarray(phase_sig, dtype=float)
    amp_sig = np.asarray(amp_sig, dtype=float)
    trial_pac_z = np.asarray(trial_pac_z, dtype=float)
    hit_miss_labels = np.asarray(hit_miss_labels, dtype=object)

    if phase_sig.ndim != 2 or amp_sig.ndim != 2:
        return None
    if phase_sig.shape != amp_sig.shape:
        return None
    if phase_sig.shape[0] != len(trial_pac_z):
        return None

    nyquist = sfreq / 2.0
    lf_low = max(float(f_pha_band[0]), 0.5)
    lf_high = min(float(f_pha_band[1]), nyquist - 1.0)
    hf_low = max(float(f_amp_band[0]), 1.0)
    hf_high = min(float(f_amp_band[1]), nyquist - 1.0)

    if lf_high <= lf_low or hf_high <= hf_low:
        return None

    high_mask = compute_high_pac_mask(
        trial_pac_z=trial_pac_z,
        hit_miss=hit_miss_labels,
        high_pac_quantile=high_pac_quantile,
        min_high_trials=min_high_trials,
    )

    cond_indices = {
        cond: np.where(high_mask & (hit_miss_labels == cond) & np.isfinite(trial_pac_z))[0]
        for cond in ('hit', 'miss')
    }

    if all(len(v) == 0 for v in cond_indices.values()):
        return None

    phase_bins = np.linspace(-np.pi, np.pi, 19)
    phase_centers = (phase_bins[:-1] + phase_bins[1:]) / 2.0

    fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex='col')

    for row_idx, cond in enumerate(('hit', 'miss')):
        idx = cond_indices[cond]
        ax_lf, ax_hf, ax_phase = axes[row_idx]

        if len(idx) == 0:
            ax_lf.text(0.5, 0.5, f'No high-PAC {cond} trials', ha='center', va='center')
            ax_hf.text(0.5, 0.5, f'No high-PAC {cond} trials', ha='center', va='center')
            ax_phase.text(0.5, 0.5, f'No high-PAC {cond} trials', ha='center', va='center')
            for ax in (ax_lf, ax_hf, ax_phase):
                ax.set_xticks([])
                ax.set_yticks([])
            continue

        lf_data = mne.filter.filter_data(
            phase_sig[idx],
            sfreq=sfreq,
            l_freq=lf_low,
            h_freq=lf_high,
            verbose=False,
        )
        hf_data = mne.filter.filter_data(
            amp_sig[idx],
            sfreq=sfreq,
            l_freq=hf_low,
            h_freq=hf_high,
            verbose=False,
        )

        hf_env = np.abs(hilbert(hf_data, axis=-1))
        lf_phase = np.angle(hilbert(lf_data, axis=-1))

        mean_lf = np.nanmean(lf_data, axis=0)
        mean_hf_env = np.nanmean(hf_env, axis=0)

        phase_flat = lf_phase.ravel()
        env_flat = hf_env.ravel()
        phase_bin_idx = np.digitize(phase_flat, phase_bins) - 1
        phase_profile = []
        for b in range(len(phase_bins) - 1):
            m = phase_bin_idx == b
            phase_profile.append(np.nanmean(env_flat[m]) if np.any(m) else np.nan)
        phase_profile = np.asarray(phase_profile, dtype=float)

        clr = '#2ecc71' if cond == 'hit' else '#e74c3c'
        ax_lf.plot(times, mean_lf, color=clr, linewidth=1.5)
        ax_lf.set_title(f'{cond.upper()} LF ({lf_low:.1f}-{lf_high:.1f} Hz) | n={len(idx)}')
        ax_lf.set_ylabel('LF signal')

        ax_hf.plot(times, mean_hf_env, color=clr, linewidth=1.5)
        ax_hf.set_title(f'{cond.upper()} HF envelope ({hf_low:.1f}-{hf_high:.1f} Hz)')
        ax_hf.set_ylabel('HF envelope')

        ax_phase.plot(phase_centers, phase_profile, color=clr, linewidth=1.5)
        ax_phase.set_title(f'{cond.upper()} phase-binned HF envelope')
        ax_phase.set_xlabel('LF phase (rad)')
        ax_phase.set_ylabel('Mean HF envelope')

    axes[1, 0].set_xlabel('Time (s)')
    axes[1, 1].set_xlabel('Time (s)')

    fig.suptitle(f'{title_prefix} | High z-scored PAC trials', fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.96])

    fig.savefig(out_file, dpi=200, bbox_inches='tight')
    plt.close(fig)
    return out_file


In [ ]:
"""
Within-region (within-electrode) PAC main loop with hit/miss labels and high-PAC plots
"""
# Analysis parameters
n_perm = 250
buf_ms = 1000
high_pac_quantile = 0.9
min_high_trials = 5

within_roi_list = [
    'ACC'
]

pac_obj_within = Pac(idpac=(2, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

all_within_dfs = []
summary_rows = []
subjects_to_process = subj_ids

print('\n' + '=' * 70)
print('Starting WITHIN-REGION (within-electrode) PAC computation')
print('Subjects:', len(subjects_to_process))
print('Events:', analysis_evs)
print('ROIs:', within_roi_list)
print('=' * 70 + '\n')

for subj_idx, subj_id in enumerate(subjects_to_process):
    print(f"[{subj_idx + 1}/{len(subjects_to_process)}] Subject: {subj_id}")
    subj_info = subj_df[subj_df.MSSMCode.astype(str) == str(subj_id)].iloc[0]

    elec_df_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df'
    if not os.path.exists(elec_df_path):
        print('  Missing electrode table:', elec_df_path)
        continue
    elec_df = pd.read_csv(elec_df_path)

    for event in analysis_evs:
        save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
        epochs_file = f'{save_path}/{event}-epo.fif'
        if not os.path.exists(epochs_file):
            print('  Missing epochs file:', epochs_file)
            continue

        print('  Event:', event)
        epochs_reref = mne.read_epochs(epochs_file, preload=True)
        sfreq = float(epochs_reref.info['sfreq'])
        edges = int(buf_ms * (sfreq / 1000.0))

        trial_map = build_hit_miss_trial_map(
            combined_df=combined_df,
            subj_id=str(subj_id),
            n_epochs=len(epochs_reref),
            day=day,
        )
        if len(trial_map) > 0:
            trial_map = trial_map.copy()
            trial_map['trials'] = trial_map['epoch_idx'] + 1
        else:
            trial_map = pd.DataFrame(columns=['participant', 'trials', 'hits', 'hit_miss', 'alignment_valid'])

        for roi in within_roi_list:
            print(f'    ROI: {roi}')
            pwise_dfs = []

            for hemi in ['l', 'r']:
                roi_elecs = elec_df[(elec_df['salman_region'] == roi) & (elec_df['hemisphere'] == hemi)]
                if len(roi_elecs) == 0:
                    continue

                electrode_names = [ch for ch in roi_elecs['label'].tolist() if ch in epochs_reref.ch_names]
                if len(electrode_names) == 0:
                    continue

                print(f'      {hemi.upper()} hemisphere electrodes: {len(electrode_names)}')

                for ch_name in electrode_names:
                    pac_file = f'{save_dir_data}/{subj_id}_{roi}_WITHIN_{ch_name}_pac.npy'

                    elec_sig = np.squeeze(epochs_reref.copy().pick(ch_name).get_data())

                    z_scored_pac, _ = compute_pac_for_pair(
                        pac_obj=pac_obj_within,
                        ph_sig=elec_sig,
                        amp_sig=elec_sig,
                        sfreq=sfreq,
                        edges=edges,
                        n_perm=n_perm,
                        random_state=0,
                    )
                    np.save(pac_file, z_scored_pac)

                    trial_pac_z = np.nanmean(z_scored_pac, axis=(0, 1))

                    pwise_df = create_pac_dataframe(
                        z_scored_pac=z_scored_pac,
                        subj_id=str(subj_id),
                        subj_info=subj_info,
                        phase_roi=roi,
                        amp_roi=roi,
                        event=event,
                        phase_freq=phase_frequency,
                        amp_freq=amp_frequency,
                        ph_ch_name=ch_name,
                        amp_ch_name=ch_name,
                        hemi=hemi,
                        learn_df=learn_df,
                    )

                    merge_cols = ['participant', 'trials', 'hits', 'hit_miss', 'alignment_valid']
                    if len(trial_map) > 0:
                        pwise_df = pwise_df.merge(trial_map[merge_cols], on=['participant', 'trials'], how='left')
                    else:
                        pwise_df['hits'] = np.nan
                        pwise_df['hit_miss'] = np.nan
                        pwise_df['alignment_valid'] = False

                    pwise_df['trial_pac_z'] = trial_pac_z
                    pwise_df['is_high_pac'] = compute_high_pac_mask(
                        trial_pac_z=pwise_df['trial_pac_z'].to_numpy(dtype=float),
                        hit_miss=pwise_df['hit_miss'].to_numpy(),
                        high_pac_quantile=high_pac_quantile,
                        min_high_trials=min_high_trials,
                    )

                    summary = summarize_hit_miss_pair(pwise_df)
                    for key, val in summary.items():
                        pwise_df[key] = val

                    summary_rows.append({
                        'participant': str(subj_id),
                        'event': event,
                        'phase_roi': roi,
                        'amp_roi': roi,
                        'phase_channel': ch_name,
                        'amp_channel': ch_name,
                        'hemi': hemi,
                        **summary,
                    })

                    plot_file = f'{save_dir_plots}/pac_highz_within_{subj_id}_{event}_{roi}_{ch_name}.png'
                    plot_high_pac_lfp_phase_locking(
                        phase_sig=elec_sig,
                        amp_sig=elec_sig,
                        trial_pac_z=pwise_df['trial_pac_z'].to_numpy(dtype=float),
                        hit_miss_labels=pwise_df['hit_miss'].to_numpy(),
                        sfreq=sfreq,
                        times=epochs_reref.times,
                        f_pha_band=(f_pha_start, f_pha_end),
                        f_amp_band=(f_amp_start, f_amp_end),
                        out_file=plot_file,
                        title_prefix=f'{subj_id} {event} {roi} within-electrode {ch_name}',
                        high_pac_quantile=high_pac_quantile,
                        min_high_trials=min_high_trials,
                    )

                    pwise_dfs.append(pwise_df)

            if len(pwise_dfs) > 0:
                all_df_pt = pd.concat(pwise_dfs, ignore_index=True)
                output_file = f'{save_dir_data}/{subj_id}_{roi}_WITHIN_pac_{event}.csv'
                all_df_pt.to_csv(output_file, index=False)
                all_within_dfs.append(all_df_pt)
                print('      Saved rows:', len(all_df_pt), '|', output_file)

if len(all_within_dfs) > 0:
    all_within_results_df = pd.concat(all_within_dfs, ignore_index=True)
    combined_output = f'{save_dir_data}/all_subjects_withinelectrode_pac_combined.csv'
    all_within_results_df.to_csv(combined_output, index=False)
    print('Saved combined trial-level:', combined_output, '| rows:', len(all_within_results_df))

    summary_df = pd.DataFrame(summary_rows)
    summary_output = f'{save_dir_data}/all_subjects_withinelectrode_pac_hitmiss_summary.csv'
    summary_df.to_csv(summary_output, index=False)
    print('Saved hit/miss summary:', summary_output, '| rows:', len(summary_df))
else:
    print('No within-region PAC results to save.')


## Output Checks

In [ ]:
"""
Load and inspect combined exports
"""
trial_file = f'{save_dir_data}/all_subjects_withinelectrode_pac_combined.csv'
summary_file = f'{save_dir_data}/all_subjects_withinelectrode_pac_hitmiss_summary.csv'

if os.path.exists(trial_file):
    trial_df = pd.read_csv(trial_file)
    print('Trial-level rows:', len(trial_df))
    expected_cols = ['hit_miss', 'hits', 'alignment_valid', 'trial_pac_z', 'is_high_pac']
    print('Required columns present:', {c: (c in trial_df.columns) for c in expected_cols})
    print('Unique hit_miss labels:', sorted(trial_df['hit_miss'].dropna().astype(str).unique().tolist()) if 'hit_miss' in trial_df.columns else [])
else:
    print('Missing trial export:', trial_file)

if os.path.exists(summary_file):
    summary_df = pd.read_csv(summary_file)
    print('Summary rows:', len(summary_df))
    display(summary_df.head())
else:
    print('Missing summary export:', summary_file)


## Hit vs miss PAC by RPE category — paired tests

Barplot: mean `trial_pac_z` by `rpe_cat` with hue = hit/miss. **Paired t-tests** below pair observations **by participant**: for each subject and RPE bin we compare mean PAC on hit trials vs mean PAC on miss trials (only subjects with both in that bin). This matches the comparison implied by the barplot’s within-bin hit vs miss contrast while respecting the repeated-measures structure.

If you instead need pairing at the electrode level `(participant, phase_channel)`, use the optional block in the code cell.

In [ ]:
"""
Paired hit vs miss tests within each rpe_cat (pairing by participant).
Requires: alignment_valid, trial_pac_z, hit_miss (or hits), and rpe_cat or rpe.
"""
import seaborn as sns
from scipy.stats import ttest_rel, wilcoxon

# --- load in-memory frame if present, else from combined CSV ---
_trial_path = f'{save_dir_data}/all_subjects_withinelectrode_pac_combined.csv'
if 'all_within_results_df' in globals() and len(all_within_results_df):
    _df = all_within_results_df.copy()
elif 'trial_df' in globals() and trial_df is not None and len(trial_df):
    _df = trial_df.copy()
elif os.path.exists(_trial_path):
    _df = pd.read_csv(_trial_path)
else:
    raise FileNotFoundError(f'Run PAC export first or set path. Missing: {_trial_path}')

_df = _df[_df.get('alignment_valid', True) == True].copy()

# hit / miss label (prefer hit_miss from trial map)
if 'hit_miss' in _df.columns and _df['hit_miss'].notna().any():
    _df = _df[_df['hit_miss'].isin(['hit', 'miss'])].copy()
    _df['hm'] = _df['hit_miss']
else:
    _df['hm'] = np.where(_df['hits'].astype(bool), 'hit', 'miss')

# RPE category: set externally or derive from rpe
if 'rpe_cat' not in _df.columns:
    if 'rpe' not in _df.columns:
        raise ValueError('Need column rpe_cat or rpe to define bins.')
    _df['rpe_cat'] = pd.qcut(
        _df['rpe'], q=3, labels=['rpe_low', 'rpe_mid', 'rpe_high'], duplicates='drop'
    )

# Barplot (same structure as your snippet; hue uses bool hits for legend if present)
_plot = _df.copy()
if 'hits' in _plot.columns:
    _plot['hits'] = _plot['hits'].astype(bool)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=_plot, x='rpe_cat', y='trial_pac_z', hue='hits', errorbar='se', ax=ax)
ax.set_title('trial_pac_z by RPE category × hit/miss (trial-level SE)')
plt.tight_layout()


def _holm(pvals):
    """Holm–Bonferroni adjustment (sorted p-values, enforce monotonicity)."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    if m == 0:
        return np.array([])
    order = np.argsort(pvals)
    sorted_p = pvals[order]
    q = np.empty(m)
    for i in range(m):
        q[i] = (m - i) * sorted_p[i]
    for i in range(m - 2, -1, -1):
        q[i] = min(q[i], q[i + 1])
    q = np.clip(q, 0.0, 1.0)
    out = np.empty(m)
    out[order] = q
    return out


def paired_hit_miss_tests(df, group_col='rpe_cat', pair_keys=('participant',), value_col='trial_pac_z'):
    """Per level of group_col: paired t-test and Wilcoxon on paired means indexed by pair_keys."""
    rows = []
    cats = [c for c in df[group_col].dropna().unique()]
    for cat in cats:
        sub = df[df[group_col] == cat]
        hit = sub[sub['hm'] == 'hit'].groupby(list(pair_keys))[value_col].mean()
        miss = sub[sub['hm'] == 'miss'].groupby(list(pair_keys))[value_col].mean()
        common = hit.index.intersection(miss.index)
        n = len(common)
        if n < 2:
            rows.append({
                group_col: cat, 'n_pairs': n,
                't_stat': np.nan, 'p_t': np.nan, 'p_wilcoxon': np.nan,
                'mean_hit_minus_miss': np.nan,
            })
            continue
        h = hit.loc[common].to_numpy(dtype=float)
        m_ = miss.loc[common].to_numpy(dtype=float)
        diff = h - m_
        t_stat, p_t = ttest_rel(h, m_)
        try:
            _, p_w = wilcoxon(diff, zero_method='wilcox', alternative='two-sided')
        except ValueError:
            p_w = np.nan
        rows.append({
            group_col: cat, 'n_pairs': n,
            't_stat': float(t_stat), 'p_t': float(p_t), 'p_wilcoxon': float(p_w) if np.isfinite(p_w) else np.nan,
            'mean_hit_minus_miss': float(np.mean(diff)),
        })
    out = pd.DataFrame(rows)
    if len(out):
        out['p_t_holm'] = _holm(out['p_t'].to_numpy())
        pw = out['p_wilcoxon'].to_numpy(dtype=float)
        mask = np.isfinite(pw)
        p_adj_w = np.full_like(pw, np.nan, dtype=float)
        if mask.any():
            p_adj_w[mask] = _holm(pw[mask])
        out['p_w_holm'] = p_adj_w
    return out


paired_subj = paired_hit_miss_tests(_df, group_col='rpe_cat', pair_keys=('participant',))
print('Paired by participant (mean trial_pac_z hit vs miss within each rpe_cat):')
display(paired_subj)

# Optional: pair by electrode — use if your question is within-electrode hit vs miss
if 'phase_channel' in _df.columns:
    paired_elec = paired_hit_miss_tests(
        _df, group_col='rpe_cat', pair_keys=('participant', 'phase_channel'),
    )
    print('Paired by (participant, phase_channel):')
    display(paired_elec)